# Run Shoujen SFT Packed HF

Audience:
- 用于快速验证本地导出的 `runs/sft-packed-hf` Transformers 模型。

Prerequisites:
- 当前 Python 环境已安装 `torch`、`transformers`、`safetensors`。
- 模型目录存在：`/Users/ding/Projects/shoujen-llm/runs/sft-packed-hf`。

Learning goals:
- 加载本地 `model.safetensors` 模型和 tokenizer。
- 用 Shoujen 训练时的 ChatML 格式构造 prompt。
- 运行一次交互式风格的生成。

## Outline

1. 检查模型目录和设备
2. 加载 tokenizer 和模型
3. 构造 chat prompt
4. 运行生成
5. 调整采样参数

In [ ]:
from __future__ import annotations

from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_DIR = Path("/Users/ding/Projects/shoujen-llm/runs/sft-packed-hf")
assert MODEL_DIR.exists(), f"Model directory not found: {MODEL_DIR}"
assert (MODEL_DIR / "model.safetensors").exists(), "model.safetensors not found"

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    DTYPE = torch.bfloat16
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    DTYPE = torch.float32
else:
    DEVICE = torch.device("cpu")
    DTYPE = torch.float32

MODEL_DIR, DEVICE, DTYPE

## Load The Model

导出目录包含 `auto_map` 和本地 remote-code 文件，所以这里使用 `trust_remote_code=True`。

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_DIR,
    trust_remote_code=True,
    use_fast=False,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    trust_remote_code=True,
    dtype=DTYPE,
)
model.to(DEVICE)
model.eval()

print(type(model).__name__)
print(f"parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M")
print(f"vocab_size: {len(tokenizer)}")

## Chat Prompt Format

SFT 数据使用的是 `<im_start>{role}\n{content}<im_end>\n` 格式。下面这个 renderer 和训练 tokenizer 的 `encode_chat(..., add_generation_prompt=True)` 保持一致。

In [ ]:
def render_chat(messages: list[dict[str, str]], *, add_generation_prompt: bool = True) -> str:
    chunks: list[str] = []
    for message in messages:
        role = message["role"]
        content = message["content"]
        chunks.append(f"<im_start>{role}\n{content}<im_end>\n")
    if add_generation_prompt:
        chunks.append("<im_start>assistant\n")
    return "".join(chunks)


sample_messages = [
    {"role": "system", "content": "汝乃ShoujenLLM，由Boris训练之文言AI助手\nLet's think step by step"},
    {"role": "user", "content": "用两句话介绍你自己。"},
]

print(render_chat(sample_messages))

## Generate A Reply

默认参数偏稳：`temperature=0.7`、`top_p=0.9`。如果要确定性输出，把 `temperature=0`。

In [ ]:
@torch.inference_mode()
def chat_once(
    user_message: str,
    *,
    system: str | None = "汝乃ShoujenLLM，由Boris训练之文言AI助手\nLet's think step by step",
    history: list[dict[str, str]] | None = None,
    max_new_tokens: int = 256,
    temperature: float = 0.1,
    top_p: float = 0.9,
    top_k: int = 50,
) -> str:
    messages: list[dict[str, str]] = []
    if system:
        messages.append({"role": "system", "content": system})
    if history:
        messages.extend(history)
    messages.append({"role": "user", "content": user_message})

    prompt = render_chat(messages, add_generation_prompt=True)
    inputs = tokenizer(prompt, add_special_tokens=False, return_tensors="pt")
    inputs = {key: value.to(DEVICE) for key, value in inputs.items()}

    eos_token_ids = [tokenizer.eos_token_id]
    im_end_token_id = getattr(model.config, "im_end_token_id", None)
    if im_end_token_id is not None:
        eos_token_ids.append(im_end_token_id)
    eos_token_ids = [token_id for token_id in eos_token_ids if token_id is not None]

    generate_kwargs = {
        "max_new_tokens": max_new_tokens,
        "eos_token_id": eos_token_ids,
        "pad_token_id": tokenizer.pad_token_id or tokenizer.eos_token_id,
    }
    if temperature <= 0:
        generate_kwargs["do_sample"] = False
    else:
        generate_kwargs.update(
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
        )

    output_ids = model.generate(**inputs, **generate_kwargs)
    new_ids = output_ids[0, inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True).strip()


reply = chat_once("天为什么是蓝色的？", max_new_tokens=128)
print(reply)

## Try Your Own Prompt

修改下面的 `question`，重新运行 cell。长回答可以调大 `max_new_tokens`。

In [ ]:
question = "解释一下 RWKV 和 Attention 的主要区别。"
print(chat_once(question, max_new_tokens=384, temperature=0.7, top_p=0.9))

## Deterministic Mode

排查模型能力或比较 prompt 时，建议先用贪心解码：`temperature=0`。

In [ ]:
print(chat_once("给我一个三步调试训练 loss 爆炸的清单。", temperature=0, max_new_tokens=256))

## Common Pitfalls

- 如果 `AutoModelForCausalLM` 提示 remote code，确认传了 `trust_remote_code=True`。
- 如果输出乱码或格式不对，确认 tokenizer 来自同一个导出目录。
- 如果显存不够，先降低 `max_new_tokens`，或在 CUDA 上把 `DTYPE` 改为 `torch.float16`。
- 如果你在容器里运行，把 `MODEL_DIR` 改成 `/workspace/shoujen-llm/runs/sft-packed-hf`。